Дані

In [1]:
import os, re, glob, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import QuantileTransformer

os.makedirs("results", exist_ok=True)
dev = torch.device("cuda")

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"не знайшов {name}: перевір Add Input"
    return hits[0]

tr = pd.read_csv(find("train_transaction.csv"))
idn = pd.read_csv(find("train_identity.csv"))
for c in tr.select_dtypes("float64").columns:
    tr[c] = tr[c].astype("float32")
df = tr.merge(idn, on="TransactionID", how="left")
df["has_identity"] = df["TransactionID"].isin(idn["TransactionID"])
df["day"] = (df["TransactionDT"] // 86400).astype(int)
df["week"] = df["day"] // 7
del tr

N = len(df)
w = df.week.values
m_tr, m_va, m_te = (w <= 17), (w >= 18) & (w <= 21), (w >= 22)
assert (m_tr.sum(), m_va.sum(), m_te.sum()) == (431125, 77090, 82325)
# розбиття для гібрида
n_tr1 = w <= 6                     # навчання M1
n_va1 = (w >= 7) & (w <= 8)        # ранній зупин M1
x_tr = (w >= 9) & (w <= 17)        # навчання XGBoost (M1 цих рядків не бачила)
x_va, x_te = m_va, m_te
assert (n_tr1.sum(), n_va1.sum(), x_tr.sum()) == (189635, 40271, 201219)
assert not (n_tr1 & x_tr).any() and not (n_va1 & x_tr).any() and w[n_va1].max() < w[x_tr].min()

split_of = np.where(m_tr, 0, np.where(m_va, 1, 2))
ts, tid = df.TransactionDT.values, df.TransactionID.values
order = np.lexsort((tid, ts))
rank = np.empty(N, dtype=np.int64); rank[order] = np.arange(N)
y = df.isFraud.values
assert y.sum() == 20663

X = df.drop(columns=["TransactionID", "isFraud", "TransactionDT", "day", "week"]).copy()
X["hour"] = (df.TransactionDT // 3600) % 24
X["weekday"] = (df.TransactionDT // 86400) % 7
X["has_identity"] = X["has_identity"].astype("int8")
for c in X.select_dtypes(include=["object", "category"]).columns:
    codes, _ = pd.factorize(X[c])
    X[c] = np.where(codes == -1, np.nan, codes).astype("float32")
df["D1n"] = (df["D1"] - df["TransactionDT"] / 86400).round()   # після X, у X не потрапляє

def metrics(y_true, p):
    k = int(y_true.sum()); o = np.argsort(-p)
    top1 = o[: max(1, int(0.01 * len(y_true)))]
    return {"pr_auc": average_precision_score(y_true, p), "roc_auc": roc_auc_score(y_true, p),
            "recall@k": y_true[o[:k]].sum() / k, "precision@1%": y_true[top1].mean()}

major = int(xgb.__version__.split(".")[0])
DEV = dict(tree_method="hist", device="cuda") if major >= 2 else dict(tree_method="gpu_hist")
print("готово:", X.shape)

готово: (590540, 434)


Граф u2, u3, u5, мітки сусідів, перевірка

In [2]:
def build_edges(cols, K, W_days):
    valid = df[cols].notna().all(axis=1).values
    idx = np.flatnonzero(valid); sub = df.loc[valid, cols]
    S, D = [], []
    for pos in sub.groupby(cols, sort=False).indices.values():
        if len(pos) < 2: continue
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt = ts[rows]; n = len(rows)
        lo = np.maximum(np.arange(n) - K, np.searchsorted(tt, tt - W_days * 86400, side="left"))
        cnt = np.arange(n) - lo; tot = int(cnt.sum())
        if tot == 0: continue
        dst_l = np.repeat(np.arange(n), cnt)
        start = np.repeat(np.cumsum(cnt) - cnt, cnt)
        src_l = np.arange(tot) - start + np.repeat(lo, cnt)
        S.append(rows[src_l].astype(np.int32)); D.append(rows[dst_l].astype(np.int32))
    return np.concatenate(S), np.concatenate(D)

REL = {"u2": (["card1", "addr1"], 50, 30),
       "u3": (["card1", "addr1", "P_emaildomain"], 50, 30),
       "u5": (["card1", "addr1", "P_emaildomain", "D1n"], 50, 30)}
R_NAMES = list(REL); KMAX = {k: v[1] for k, v in REL.items()}
edges = {k: build_edges(*v) for k, v in REL.items()}
for k, (s, d) in edges.items():
    assert (rank[s] < rank[d]).all() and (s != d).all()
    assert (split_of[s[split_of[d] == 0]] == 0).all()
    print(k, f"ребер {len(s):,}")

NBR, DEG = [], []
for rn in R_NAMES:
    s, d = edges[rn]
    o = np.argsort(d, kind="stable"); ds, ss = d[o], s[o]
    cnt = np.bincount(d, minlength=N); start = np.cumsum(cnt) - cnt
    pos = np.arange(len(ds)) - start[ds]
    assert pos.max() < KMAX[rn]
    tab = np.full((N, KMAX[rn]), -1, dtype=np.int32); tab[ds, pos] = ss
    NBR.append(torch.from_numpy(tab).to(dev))
    DEG.append(torch.from_numpy(cnt.astype(np.int64)).to(dev))

DL, W_LAB = 7, 30
def label_feats(cols, W_days, DLd):
    valid = df[cols].notna().all(axis=1).values; idx = np.flatnonzero(valid)
    nk = np.full(N, np.nan, "float32"); fk = np.full(N, np.nan, "float32")
    for pos in df.loc[valid, cols].groupby(cols, sort=False).indices.values():
        rows = idx[pos]; rows = rows[np.argsort(rank[rows], kind="stable")]
        tt, ff = ts[rows], y[rows]; cs = np.concatenate([[0], np.cumsum(ff)])
        a = np.searchsorted(tt, tt, side="left")
        hi = np.minimum(np.searchsorted(tt, tt - DLd * 86400, side="right"), a)
        lo = np.minimum(np.searchsorted(tt, tt - W_days * 86400, side="left"), hi)
        nk[rows] = hi - lo; fk[rows] = cs[hi] - cs[lo]
    return nk, fk

LF = {}
for rel, (cols, _, _) in REL.items():
    nk, fk = label_feats(cols, W_LAB, DL)
    LF[f"{rel}_nk"], LF[f"{rel}_fk"] = nk, fk
    LF[f"{rel}_rate"] = (fk / np.where(nk > 0, nk, np.nan)).astype("float32")
LF = pd.DataFrame(LF)

rng = np.random.default_rng(2); bad = 0
for rel, (cols, _, _) in REL.items():
    cand = np.flatnonzero(df[cols].notna().all(axis=1).values)
    for i in rng.choice(cand, 60, replace=False):
        same = np.ones(N, bool)
        for c in cols: same &= (df[c].values == df[c].values[i])
        kn = same & (ts <= ts[i] - DL * 86400) & (ts >= ts[i] - W_LAB * 86400)
        if (int(kn.sum()) != int(LF[f"{rel}_nk"].values[i]) or
                int(y[kn].sum()) != int(LF[f"{rel}_fk"].values[i])): bad += 1
print("розбіжностей з перерахунком «в лоб»:", bad); assert bad == 0
BASE = pd.concat([X, LF], axis=1)
print("граф і мітки готові; BASE:", BASE.shape)

u2 ребер 12,260,904
u3 ребер 5,295,877
u5 ребер 478,008
розбіжностей з перерахунком «в лоб»: 0
граф і мітки готові; BASE: (590540, 443)


Вхід нейромережі

In [3]:
Xn = X.values.astype("float32")
grp = lambda p: [i for i, c in enumerate(X.columns) if re.fullmatch(p, c)]
nanf = np.stack([np.isnan(Xn[:, g]).mean(1) for g in
                 (grp(r"V\d+"), grp(r"D\d+"), grp(r"id_\d+"), grp(r"M\d+"))], 1).astype("float32")
qt = QuantileTransformer(n_quantiles=200, output_distribution="normal",
                         subsample=200_000, random_state=0).fit(Xn[n_tr1])
Xt = np.hstack([np.nan_to_num(qt.transform(Xn), nan=0.0).clip(-5, 5).astype("float32"), nanf])
cols_ = []
for r in R_NAMES:
    g = lambda k, r=r: np.nan_to_num(LF[f"{r}_{k}"].values)
    cols_ += [np.log1p(g("nk")), np.log1p(g("fk")), g("rate"), (g("nk") > 0).astype("float32")]
M = np.stack(cols_, 1).astype("float32")
M = (M - M[n_tr1].mean(0)) / (M[n_tr1].std(0) + 1e-6)
XG = torch.from_numpy(np.hstack([Xt, M])).half().to(dev)
print("вхід нейромережі:", tuple(XG.shape))

вхід нейромережі: (590540, 450)


Моделі (з поверненням ембедингів) і тест на витік

In [4]:
def sample(nodes, r, s, gen=None):
    deg = DEG[r][nodes]
    u = torch.rand(nodes.shape[0], s, device=dev, generator=gen)
    idx = (u * deg.unsqueeze(1)).long().clamp_(max=KMAX[R_NAMES[r]] - 1)
    nb = NBR[r][nodes.unsqueeze(1), idx].long()
    valid = (deg > 0).unsqueeze(1).expand(-1, s) & (nb >= 0)
    return nb.clamp(min=0), valid

def hops(tgt, s1, s2, gen=None):
    B, R = tgt.shape[0], len(R_NAMES)
    n1, v1 = zip(*[sample(tgt, r, s1, gen) for r in range(R)])
    N1, M1 = torch.stack(n1, 1), torch.stack(v1, 1)
    n2, v2 = zip(*[sample(N1.reshape(-1), r, s2, gen) for r in range(R)])
    N2 = torch.stack(n2, 1).view(B, R, s1, R, s2)
    M2 = torch.stack(v2, 1).view(B, R, s1, R, s2) & M1[:, :, :, None, None]
    return N1, M1, N2, M2

class RelSAGE2(nn.Module):
    def __init__(self, F_in, H=128, R=3, drop=0.2):
        super().__init__()
        self.R = R
        self.enc = nn.Sequential(nn.Linear(F_in, H), nn.ReLU(), nn.Dropout(drop))
        self.W1s, self.W2s = nn.Linear(H, H), nn.Linear(H, H)
        self.W1n = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
        self.W2n = nn.ModuleList([nn.Linear(H, H, bias=False) for _ in range(R)])
        self.out = nn.Sequential(nn.Dropout(drop), nn.Linear(H, 1))

    @staticmethod
    def mmean(e, m, dim):
        m = m.unsqueeze(-1).to(e.dtype)
        return (e * m).sum(dim) / m.sum(dim).clamp(min=1)

    def forward(self, XG, tgt, s1, s2, gen=None, emb=False):
        B, R = tgt.shape[0], self.R
        N1, M1, N2, M2 = hops(tgt, s1, s2, gen)
        enc = lambda ids: self.enc(XG[ids].float())
        e_t = enc(tgt)
        e1 = enc(N1.reshape(-1)).view(B, R, s1, -1)
        e2 = enc(N2.reshape(-1)).view(B, R, s1, R, s2, -1)
        a2 = self.mmean(e2, M2, 4)
        h1_u = F.relu(self.W1s(e1) + sum(self.W1n[j](a2[:, :, :, j]) for j in range(R)))
        a1 = self.mmean(e1, M1, 2)
        h1_t = F.relu(self.W1s(e_t) + sum(self.W1n[j](a1[:, j]) for j in range(R)))
        ah = self.mmean(h1_u, M1, 2)
        h2 = F.relu(self.W2s(h1_t) + sum(self.W2n[j](ah[:, j]) for j in range(R)))
        lg = self.out(h2).squeeze(-1)
        return (lg, h2) if emb else lg

class MLPNet(nn.Module):
    def __init__(self, F_in, H=128, drop=0.2):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(F_in, H), nn.ReLU(), nn.Dropout(drop),
                                  nn.Linear(H, H), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(drop), nn.Linear(H, 1))
    def forward(self, XG, tgt, s1=None, s2=None, gen=None, emb=False):
        h = self.body(XG[tgt].float()); lg = self.head(h).squeeze(-1)
        return (lg, h) if emb else lg

sp_t = torch.as_tensor(split_of, device=dev); t_t = torch.as_tensor(ts, device=dev)
rg = np.random.default_rng(0)
for sp, cap in [(0, 0), (1, 1)]:
    ids = torch.as_tensor(rg.choice(np.flatnonzero(split_of == sp), 4000, replace=False), device=dev)
    N1, M1, N2, M2 = hops(ids, 10, 5)
    assert (sp_t[N1[M1]] <= cap).all() and (sp_t[N2[M2]] <= cap).all(), "сусіди з майбутньої частини"
    assert (t_t[N1] <= t_t[ids][:, None, None])[M1].all(), "сусід пізніший за вузол (крок 1)"
    assert (t_t[N2] <= t_t[N1][:, :, :, None, None])[M2].all(), "сусід пізніший за вузол (крок 2)"
print("тест на витік пройдено")

тест на витік пройдено


Гібридний конвеєр, один seed

In [5]:
@torch.no_grad()
def predict(model, idx_np, s1, s2, seed, bs=2048):
    model.eval(); ids = torch.as_tensor(idx_np, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(1234 + 100 * seed)
    return torch.cat([torch.sigmoid(model(XG, ids[i:i + bs], s1, s2, gen)).float().cpu()
                      for i in range(0, len(ids), bs)]).numpy()

def train_m1(make_model, seed, epochs, patience, bs=1024, s1=10, s2=5, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    tr1, va1 = np.flatnonzero(n_tr1), np.flatnonzero(n_va1)
    model = make_model(XG.shape[1]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    posw = torch.tensor(float((y[tr1] == 0).sum() / (y[tr1] == 1).sum()), device=dev)
    tri = torch.as_tensor(tr1, device=dev); yt = torch.as_tensor(y, device=dev).float()
    best, bad, state = -1.0, 0, None
    for ep in range(1, epochs + 1):
        model.train(); perm = tri[torch.randperm(len(tri), device=dev)]
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(model(XG, b, s1, s2), yt[b], pos_weight=posw)
            opt.zero_grad(); loss.backward(); opt.step()
        v = average_precision_score(y[va1], predict(model, va1, s1, s2, seed))
        if v > best:
            best, bad = v, 0; state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(state); return model, best

@torch.no_grad()
def extract(model, idx_np, s1, s2, seed, bs=2048):
    model.eval(); ids = torch.as_tensor(idx_np, device=dev)
    gen = torch.Generator(device=dev); gen.manual_seed(777 + seed)
    L, E = [], []
    for i in range(0, len(ids), bs):
        lg, h = model(XG, ids[i:i + bs], s1, s2, gen, emb=True)
        L.append(lg.float().cpu()); E.append(h.float().cpu())
    return torch.cat(L).numpy(), torch.cat(E).numpy()

spw_x = (y[x_tr] == 0).sum() / (y[x_tr] == 1).sum()
def xgb_arm(Xf, seed):
    m = XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=6, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=spw_x, eval_metric="aucpr",
                      early_stopping_rounds=50, random_state=seed, **DEV)
    m.fit(Xf[x_tr], y[x_tr], eval_set=[(Xf[x_va], y[x_va])], verbose=False)
    r = metrics(y[x_te], m.predict_proba(Xf[x_te])[:, 1])
    r.update(val_aucpr=m.best_score, best_iter=m.best_iteration); return r

def one_seed(seed):
    idx_emb = np.flatnonzero(w >= 9)
    embs, diag, rows = {}, [], []
    for name, mk, cfg in [("MLP", MLPNet, dict(epochs=15, patience=4)),
                          ("GNN", RelSAGE2, dict(epochs=12, patience=3))]:
        t0 = time.time(); model, vb = train_m1(mk, seed, **cfg)
        lg, h = extract(model, idx_emb, 10, 5, seed)
        E = np.full((N, 129), np.nan, "float32"); E[idx_emb, 0] = lg; E[idx_emb, 1:] = h
        embs[name] = pd.DataFrame(E, columns=[f"{name}_logit"] + [f"{name}_e{j}" for j in range(128)])
        diag.append({"seed": seed, "модель M1": name, "val M1": vb,
                     "test PR-AUC самої M1": average_precision_score(y[x_te], E[x_te, 0]),
                     "секунд": time.time() - t0})
        print(f"seed {seed}: M1 {name} готова, val {vb:.4f}, {time.time()-t0:.0f} с", flush=True)
    arms = {"контроль (без ембедингів)": BASE,
            "+ ембединги MLP": pd.concat([BASE, embs["MLP"]], axis=1),
            "+ ембединги GNN": pd.concat([BASE, embs["GNN"]], axis=1)}
    for arm, Xf in arms.items():
        r = xgb_arm(Xf, seed); r.update(arm=arm, seed=seed); rows.append(r)
        print(f"seed {seed}: {arm}: val {r['val_aucpr']:.4f} test {r['pr_auc']:.4f}", flush=True)
    return rows, diag

rows_h, diag_h = one_seed(0)
pd.DataFrame(rows_h).to_csv("results/ieee_cis_hybrid_seed0.csv", index=False)

seed 0: M1 MLP готова, val 0.6230, 10 с
seed 0: M1 GNN готова, val 0.6004, 112 с


/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [10:10:31] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


seed 0: контроль (без ембедингів): val 0.6696 test 0.6233
seed 0: + ембединги MLP: val 0.6474 test 0.6257
seed 0: + ембединги GNN: val 0.6502 test 0.6240


Решта чотири seed і підсумок

In [6]:
for sd in range(1, 5):
    r_, d_ = one_seed(sd); rows_h += r_; diag_h += d_
    pd.DataFrame(rows_h).to_csv("results/ieee_cis_hybrid_5seeds.csv", index=False)

R = pd.DataFrame(rows_h); order_a = ["контроль (без ембедингів)", "+ ембединги MLP", "+ ембединги GNN"]
P = lambda c: R.pivot(index="seed", columns="arm", values=c)[order_a]
pt, pv = P("pr_auc"), P("val_aucpr"); ctrl = order_a[0]
S = pd.DataFrame({"val PR-AUC": pv.mean(), "test PR-AUC": pt.mean(), "std": pt.std(),
                  "Δval до контролю": pv.sub(pv[ctrl], axis=0).mean(),
                  "Δtest до контролю": pt.sub(pt[ctrl], axis=0).mean(),
                  "SE Δtest": pt.sub(pt[ctrl], axis=0).std() / np.sqrt(len(pt))}).round(4)
S["приймаємо (Δval ≥ +0,01 і Δtest > 0)"] = (S["Δval до контролю"] >= 0.01) & (S["Δtest до контролю"] > 0)
display(S); S.to_csv("results/ieee_cis_hybrid_summary.csv")
d = pt["+ ембединги GNN"] - pt["+ ембединги MLP"]
print(f"GNN-ембединги − MLP-ембединги: {d.mean():+.4f} ± {d.std()/np.sqrt(len(d)):.4f} (SE)")
display(pd.DataFrame(diag_h).groupby("модель M1")[["val M1", "test PR-AUC самої M1"]].agg(["mean", "std"]).round(3))

seed 1: M1 MLP готова, val 0.6169, 3 с
seed 1: M1 GNN готова, val 0.6046, 187 с
seed 1: контроль (без ембедингів): val 0.6662 test 0.6229
seed 1: + ембединги MLP: val 0.6579 test 0.6188
seed 1: + ембединги GNN: val 0.6498 test 0.6245
seed 2: M1 MLP готова, val 0.6262, 6 с
seed 2: M1 GNN готова, val 0.6229, 129 с
seed 2: контроль (без ембедингів): val 0.6639 test 0.6287
seed 2: + ембединги MLP: val 0.6582 test 0.6223
seed 2: + ембединги GNN: val 0.6560 test 0.6238
seed 3: M1 MLP готова, val 0.6200, 5 с
seed 3: M1 GNN готова, val 0.6218, 129 с
seed 3: контроль (без ембедингів): val 0.6593 test 0.6189
seed 3: + ембединги MLP: val 0.6529 test 0.6240
seed 3: + ембединги GNN: val 0.6520 test 0.6185
seed 4: M1 MLP готова, val 0.6263, 6 с
seed 4: M1 GNN готова, val 0.6174, 171 с
seed 4: контроль (без ембедингів): val 0.6659 test 0.6297
seed 4: + ембединги MLP: val 0.6555 test 0.6149
seed 4: + ембединги GNN: val 0.6557 test 0.6191


,val PR-AUC,test PR-AUC,std,Δval до контролю,Δtest до контролю,SE Δtest,"приймаємо (Δval ≥ +0,01 і Δtest > 0)"
arm,,,,,,,
контроль (без ембедингів),0.6650,0.6247,0.0045,0.0000,0.0000,0.0000,False
+ ембединги MLP,0.6544,0.6212,0.0043,-0.0106,-0.0035,0.0035,False
+ ембединги GNN,0.6528,0.6220,0.0029,-0.0122,-0.0027,0.0023,False


GNN-ембединги − MLP-ембединги: +0.0008 ± 0.0020 (SE)


val M1        test PR-AUC самої M1       
            mean    std                 mean    std
модель M1                                          
GNN        0.613  0.010                0.553  0.011
MLP        0.622  0.004                0.555  0.004